# Manual topic annotation

Generate a reproducible annotation sheet with five centroid-nearest and five boundary questions per final topic. This notebook makes no API calls and does not change BERTopic assignments or LLM suggestions.

In [1]:
import ast
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
HIERARCHY_DIR = PROJECT_DIR / 'results/hierarchical_topics/nn30_mcs20_seed2024'
ASSIGNMENTS_PATH = HIERARCHY_DIR / 'unique_questions_with_fine_topics.csv'
TOPICS_PATH = HIERARCHY_DIR / 'fine_topic_descriptions.csv'
SUGGESTIONS_PATH = PROJECT_DIR / 'results/topic_labeling/topic_label_suggestions.csv'
EMBEDDINGS_PATH = PROJECT_DIR / 'results/hierarchical_topics/embeddings_all_questions_google_embeddinggemma-300m.npy'
DATA_PATH = PROJECT_DIR / 'data/perguntas/relevant_question_extraction_gpt-5-4-mini_high_flex.csv'
GOLD_PATH = PROJECT_DIR / 'data/files/queries.txt'
OUTPUT_DIR = PROJECT_DIR / 'results/manual_annotation'
ANNOTATION_PATH = OUTPUT_DIR / 'topic_annotation.csv'
AUDIT_PATH = OUTPUT_DIR / 'topic_annotation_sampling_audit.csv'
VALIDATED_PATH = OUTPUT_DIR / 'topic_annotations_validated.csv'
N_CENTRAL = 5
N_BOUNDARY = 5
EXPECTED_TOPICS = 158
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
required_paths = [ASSIGNMENTS_PATH, TOPICS_PATH, SUGGESTIONS_PATH, EMBEDDINGS_PATH, DATA_PATH, GOLD_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Arquivos ausentes:\n' + '\n'.join(missing))

assignments = pd.read_csv(ASSIGNMENTS_PATH)
topics = pd.read_csv(TOPICS_PATH)
suggestions = pd.read_csv(SUGGESTIONS_PATH)
embeddings = np.load(EMBEDDINGS_PATH)
gold_questions = {line.strip() for line in GOLD_PATH.read_text(encoding='utf-8').splitlines() if line.strip()}

source = pd.read_csv(DATA_PATH)
source = source[source['perguntas'].ne('[]')].copy()
source['perguntas'] = source['perguntas'].apply(lambda value: ast.literal_eval(value) if isinstance(value, str) else value)
all_occurrences = source.explode('perguntas')['perguntas'].dropna().astype(str).reset_index(drop=True)
first_occurrence = ~all_occurrences.duplicated()
embedding_by_question = dict(zip(all_occurrences[first_occurrence], np.flatnonzero(first_occurrence)))
if len(embeddings) != len(all_occurrences):
    raise ValueError('O número de embeddings difere do número de ocorrências das perguntas.')

questions = assignments.loc[assignments['fine_topic_id'] != -1, ['perguntas', 'fine_topic_id', 'source_topic_id']].copy()
questions['perguntas'] = questions['perguntas'].astype(str)
questions = questions[~questions['perguntas'].isin(gold_questions)].drop_duplicates('perguntas').reset_index(drop=True)
questions['embedding_index'] = questions['perguntas'].map(embedding_by_question)
if questions['embedding_index'].isna().any():
    raise ValueError('Há perguntas sem embedding correspondente.')

suggestion_columns = ['topic_id', 'suggestion_1', 'suggestion_2', 'suggestion_3']
if suggestions[suggestion_columns].isna().any().any() or suggestions['topic_id'].duplicated().any():
    raise ValueError('As sugestões possuem valores ausentes ou topic_id duplicado.')
topic_words = topics.loc[topics['Topic'] != -1].set_index('Topic')['Representation'].to_dict()
regular_topic_ids = sorted(questions['fine_topic_id'].astype(int).unique())
if len(regular_topic_ids) != EXPECTED_TOPICS:
    raise ValueError(f'Esperados {EXPECTED_TOPICS} tópicos regulares, encontrados {len(regular_topic_ids)}.')
if set(regular_topic_ids) != set(suggestions['topic_id'].astype(int)):
    raise ValueError('Os IDs das sugestões não correspondem aos tópicos finais.')
print(f'{len(questions):,} perguntas elegíveis em {len(regular_topic_ids)} tópicos')

15,737 perguntas elegíveis em 158 tópicos


In [3]:
annotation_rows = []
audit_rows = []
suggestions_by_topic = suggestions.set_index('topic_id')
for topic_id in regular_topic_ids:
    frame = questions.loc[questions['fine_topic_id'] == topic_id].copy()
    if len(frame) < N_CENTRAL + N_BOUNDARY:
        raise ValueError(f'Tópico {topic_id} possui somente {len(frame)} perguntas elegíveis.')
    indices = frame['embedding_index'].astype(int).to_numpy()
    vectors = embeddings[indices]
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    centroid = vectors.mean(axis=0)
    centroid /= np.linalg.norm(centroid)
    similarities = vectors @ centroid
    central_positions = np.argsort(-similarities, kind='stable')[:N_CENTRAL]
    boundary_candidates = [position for position in np.argsort(similarities, kind='stable') if position not in set(central_positions)]
    boundary_positions = np.asarray(boundary_candidates[:N_BOUNDARY])
    selected_positions = np.concatenate([central_positions, boundary_positions])
    if len(set(selected_positions)) != N_CENTRAL + N_BOUNDARY:
        raise AssertionError(f'Amostra repetida no tópico {topic_id}.')

    central_questions = frame.iloc[central_positions]['perguntas'].tolist()
    boundary_questions = frame.iloc[boundary_positions]['perguntas'].tolist()
    topic_suggestions = suggestions_by_topic.loc[topic_id]
    row = {
        'topic_id': topic_id,
        'topic_size': len(frame),
        'top_words': topic_words.get(topic_id, ''),
        'suggestion_1': topic_suggestions['suggestion_1'],
        'suggestion_2': topic_suggestions['suggestion_2'],
        'suggestion_3': topic_suggestions['suggestion_3'],
    }
    row.update({f'central_example_{i}': question for i, question in enumerate(central_questions, 1)})
    row.update({f'boundary_example_{i}': question for i, question in enumerate(boundary_questions, 1)})
    row.update({'relevance': '', 'selected_name_option': '', 'manual_name': '', 'notes': '', 'annotator_id': '', 'annotated_at': ''})
    annotation_rows.append(row)

    for sample_type, positions in [('central', central_positions), ('boundary', boundary_positions)]:
        for rank, position in enumerate(positions, 1):
            question = frame.iloc[position]['perguntas']
            audit_rows.append({
                'topic_id': topic_id,
                'sample_type': sample_type,
                'position': rank,
                'question': question,
                'cosine_similarity': float(similarities[position]),
                'question_sha256': hashlib.sha256(question.encode()).hexdigest(),
            })

annotation = pd.DataFrame(annotation_rows).sort_values('topic_id').reset_index(drop=True)
audit = pd.DataFrame(audit_rows).sort_values(['topic_id', 'sample_type', 'position']).reset_index(drop=True)
annotation_fields = ['relevance', 'selected_name_option', 'manual_name', 'notes', 'annotator_id', 'annotated_at']
if ANNOTATION_PATH.exists():
    previous = pd.read_csv(ANNOTATION_PATH, dtype=str).set_index('topic_id')
    for field in annotation_fields:
        annotation[field] = annotation['topic_id'].astype(str).map(previous[field]).fillna('')

annotation.to_csv(ANNOTATION_PATH, index=False)
audit.to_csv(AUDIT_PATH, index=False)
print(ANNOTATION_PATH)
print(AUDIT_PATH)
display(annotation.head())

/scratch/victoria.estanislau/projeto-gravidez/results/manual_annotation/topic_annotation.csv
/scratch/victoria.estanislau/projeto-gravidez/results/manual_annotation/topic_annotation_sampling_audit.csv


,topic_id,topic_size,top_words,suggestion_1,suggestion_2,suggestion_3,central_example_1,central_example_2,central_example_3,central_example_4,...,boundary_example_2,boundary_example_3,boundary_example_4,boundary_example_5,relevance,selected_name_option,manual_name,notes,annotator_id,annotated_at
0,0,1267,"['maternidade', 'salário', 'auxílio', 'direito...",Auxílio maternidade do INSS,Salário maternidade do INSS,Direito ao auxílio maternidade INSS,Uma gestante de 6 meses que vai começar a cont...,É necessário contribuir por mais um mês para t...,Com quantos meses de contribuição é possível t...,"No meu caso, preciso aguardar minha filha nasc...",...,É permitido trabalhar como afiliado da Shopee ...,"No meu caso, a guia para continuar contribuind...",Devo preencher a competência com o mês e o ano...,"Por que, ao tentar emitir a guia pelo aplicati...",,,,,,
1,1,360,"['anticoncepcional', 'pílula', 'seguinte', 'to...",Risco de gravidez com anticoncepcional,Chance de engravidar usando anticoncepcional,Possibilidade de gravidez no uso de anticoncep...,Tive relação no 9º dia de uso do anticoncepcio...,É possível que eu esteja grávida após iniciar ...,"Tomar a pílula anticoncepcional no mesmo dia, ...",Depois de parar a injeção anticoncepcional tri...,...,"Depois de terminar essa cartela, devo pular a ...","Sertralina, clonazepam e Pasalix podem interfe...","O uso de sertralina, clonazepam e Pasalix pode...",O pantoprazol corta o efeito do anticoncepcional?,,,,,,
2,2,361,"['progesterona', 'via', 'oral', 'vaginal', 'us...",Uso de progesterona na reposição,Reposição com progesterona uso,Via e tempo de progesterona,A progesterona precisa ser usada por via oral?,A progesterona deve ser tomada de forma contínua?,É aconselhável tomar um comprimido de progeste...,Por quanto tempo a progesterona pode ser usada?,...,A tibolona é segura para uma mulher de 51 anos...,A via oral do Ultragestan pode causar náuseas?,O AD-Til deve ser administrado em 4 gotas por ...,Pode haver hipoplasia mamária mesmo com a repo...,,,,,,
3,3,308,"['ela', 'grávida', 'está', 'pessoa', 'menciona...",A pessoa mencionada está grávida,Ela realmente está grávida,A mamãe está grávida de verdade,Será que ela está mesmo grávida?,Ela estava realmente grávida?,E se ela estiver realmente grávida?,Ela está grávida de verdade?,...,Como um teste de gravidez em papel consegue di...,Como nasce uma pedra grávida?,Isso é um espermatozoide?,Maria engravidou de Jesus na adolescência também?,,,,,,
4,4,264,"['nome', 'for', 'nomes', 'vocês', 'chamar', 'q...",Escolha de nomes para bebês,Nomes sugeridos para bebê,Sugestão de nomes maternos,Qual nome seria escolhido se o bebê for menina?,Qual nome seria escolhido se o bebê for menino?,Qual nome você sugere se o bebê for menino?,Qual nome colocar se o bebê for menino?,...,Qual é o nome desse adoçante e onde posso comp...,Um bebê pode engasgar ao ser alimentado com se...,Qual apelido é mais מתאים para Valentina: Vale...,Meu segundo bebê também pode estar com APLV?,,,,,,


Fill `topic_annotation.csv` outside the notebook. Use `relevante` or `irrelevante` in `relevance`. For relevant topics, use `1`, `2`, `3`, or `manual` in `selected_name_option`. Then run the validation cell below.

In [ ]:
filled = pd.read_csv(ANNOTATION_PATH, dtype=str).fillna('')
if filled['relevance'].eq('').all():
    print('Planilha gerada e ainda não anotada. Preencha topic_annotation.csv antes de validar.')
else:
    errors = []
    allowed_relevance = {'relevante', 'irrelevante'}
    allowed_options = {'1', '2', '3', 'manual'}
    example_columns = [f'central_example_{i}' for i in range(1, 6)] + [f'boundary_example_{i}' for i in range(1, 6)]
    for row in filled.itertuples(index=False):
        relevance = row.relevance.strip().lower()
        option = row.selected_name_option.strip().lower()
        if relevance not in allowed_relevance:
            errors.append(f'Tópico {row.topic_id}: relevância inválida ou ausente.')
        if relevance == 'relevante' and option not in allowed_options:
            errors.append(f'Tópico {row.topic_id}: selecione 1, 2, 3 ou manual.')
        if relevance == 'relevante' and option == 'manual' and not row.manual_name.strip():
            errors.append(f'Tópico {row.topic_id}: nome manual ausente.')
        examples = [getattr(row, column) for column in example_columns]
        if len(examples) != len(set(examples)) or any(not example.strip() for example in examples):
            errors.append(f'Tópico {row.topic_id}: exemplos ausentes ou repetidos.')
    if errors:
        raise ValueError('Erros de validação:\n' + '\n'.join(errors))

    option_to_column = {'1': 'suggestion_1', '2': 'suggestion_2', '3': 'suggestion_3'}
    def resolve_name(row):
        if row['relevance'].strip().lower() == 'irrelevante':
            return ''
        option = row['selected_name_option'].strip().lower()
        return row['manual_name'].strip() if option == 'manual' else row[option_to_column[option]]

    validated = filled.copy()
    validated['relevance'] = validated['relevance'].str.strip().str.lower()
    validated['selected_name_option'] = validated['selected_name_option'].str.strip().str.lower()
    validated['selected_name'] = validated.apply(resolve_name, axis=1)
    validated.to_csv(VALIDATED_PATH, index=False)
    print(VALIDATED_PATH)
    display(validated[['topic_id', 'relevance', 'selected_name_option', 'selected_name']].head())